In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.default.aqi_silver")
print("Silver rows:", silver.count())

Silver rows: 28157


In [0]:
gold_city = (
    silver
    .filter(F.col("AQI").isNotNull())
    .groupBy("City")
    .agg(
        F.round(F.avg("AQI"), 1).alias("avg_aqi"),
        F.round(F.max("AQI"), 1).alias("max_aqi"),
        F.count("AQI").alias("days_with_aqi")
    )
    .orderBy(F.desc("avg_aqi"))
)

gold_city.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_city_avg_aqi")
display(gold_city)

City,avg_aqi,max_aqi,days_with_aqi
Ahmedabad,452.1,2049.0,1334
Delhi,259.5,716.0,1999
Patna,240.8,619.0,1459
Gurugram,225.1,891.0,1453
Lucknow,218.0,707.0,1893
Talcher,172.9,570.0,698
Jorapokhar,159.3,604.0,771
Brajrajnagar,150.3,355.0,713
Kolkata,140.6,475.0,754
Guwahati,140.1,956.0,495


In [0]:
ahmedabad_check = (
    silver
    .filter((F.col("City") == "Ahmedabad") & F.col("AQI").isNotNull())
    .groupBy("Year")
    .agg(
        F.round(F.avg("AQI"), 0).alias("avg_aqi"),
        F.round(F.max("AQI"), 0).alias("max_aqi"),
        F.count("AQI").alias("days_with_aqi")
    )
    .orderBy("Year")
)

display(ahmedabad_check)

Year,avg_aqi,max_aqi,days_with_aqi
2015,311.0,1247.0,263
2016,310.0,1842.0,117
2017,559.0,1747.0,69
2018,622.0,2049.0,357
2019,516.0,1719.0,352
2020,242.0,1291.0,176


In [0]:
above_500 = (
    silver
    .filter(F.col("AQI") > 500)
    .groupBy("City")
    .agg(F.count("AQI").alias("rows_above_500"))
    .orderBy(F.desc("rows_above_500"))
)

display(above_500)
print("Total rows above 500:", silver.filter(F.col("AQI") > 500).count())
print("Total rows with AQI:", silver.filter(F.col("AQI").isNotNull()).count())

City,rows_above_500
Ahmedabad,412
Delhi,48
Patna,25
Gurugram,22
Lucknow,15
Jorapokhar,7
Hyderabad,4
Amritsar,4
Talcher,4
Guwahati,2


Total rows above 500: 543
Total rows with AQI: 24850


In [0]:
gold_city = (
    silver
    .filter(F.col("AQI").isNotNull())
    .groupBy("City")
    .agg(
        F.round(F.avg("AQI"), 1).alias("avg_aqi_all"),
        F.round(F.avg(F.when(F.col("AQI") <= 500, F.col("AQI"))), 1).alias("avg_aqi_capped"),
        F.sum(F.when(F.col("AQI") > 500, 1).otherwise(0)).alias("rows_above_500"),
        F.round(F.max("AQI"), 1).alias("max_aqi"),
        F.count("AQI").alias("days_with_aqi")
    )
    .orderBy(F.desc("avg_aqi_capped"))
)

gold_city.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_city_avg_aqi")
display(gold_city)

City,avg_aqi_all,avg_aqi_capped,rows_above_500,max_aqi,days_with_aqi
Ahmedabad,452.1,291.7,412,2049.0,1334
Delhi,259.5,252.1,48,716.0,1999
Patna,240.8,235.6,25,619.0,1459
Gurugram,225.1,219.4,22,891.0,1453
Lucknow,218.0,215.2,15,707.0,1893
Talcher,172.9,170.8,4,570.0,698
Jorapokhar,159.3,155.6,7,604.0,771
Brajrajnagar,150.3,150.3,0,355.0,713
Kolkata,140.6,140.6,0,475.0,754
Guwahati,140.1,137.0,2,956.0,495


In [0]:
gold_monthly = (
    silver
    .filter(F.col("AQI").isNotNull() & (F.col("AQI") <= 500))
    .groupBy("Year", "Month", "Season")
    .agg(
        F.round(F.avg("AQI"), 1).alias("avg_aqi"),
        F.count("AQI").alias("readings"),
        F.countDistinct("City").alias("cities_reporting")
    )
    .orderBy("Year", "Month")
)

gold_monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_monthly_trend")
display(gold_monthly)

Year,Month,Season,avg_aqi,readings,cities_reporting
2015,1,Winter,337.8,33,2
2015,2,Winter,328.5,42,2
2015,3,Summer,235.1,75,6
2015,4,Summer,185.8,167,6
2015,5,Summer,187.3,182,6
2015,6,Monsoon,169.1,179,6
2015,7,Monsoon,149.6,184,6
2015,8,Monsoon,156.3,186,6
2015,9,Monsoon,165.1,177,6
2015,10,Post-Monsoon,215.9,191,7


In [0]:
gold_pollutants = (
    silver
    .groupBy("City")
    .agg(
        F.round(F.avg("PM2_5"), 1).alias("avg_pm2_5"),
        F.round(F.avg("PM10"), 1).alias("avg_pm10"),
        F.round(F.avg("NO2"), 1).alias("avg_no2"),
        F.round(F.avg("SO2"), 1).alias("avg_so2"),
        F.round(F.avg("CO"), 1).alias("avg_co"),
        F.round(F.avg("O3"), 1).alias("avg_o3"),
        F.round(F.avg("NH3"), 1).alias("avg_nh3"),
        F.count("PM2_5").alias("pm2_5_readings"),
        F.count("PM10").alias("pm10_readings")
    )
    .orderBy(F.desc("avg_pm2_5"))
)

gold_pollutants.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_pollutant_summary")
display(gold_pollutants)

City,avg_pm2_5,avg_pm10,avg_no2,avg_so2,avg_co,avg_o3,avg_nh3,pm2_5_readings,pm10_readings
Patna,123.5,126.7,37.5,22.1,1.5,37.2,18.4,1537,191
Delhi,117.2,232.8,50.8,15.9,2.0,51.3,42.0,2007,1932
Gurugram,117.1,191.5,23.4,9.4,1.3,34.4,26.1,1525,740
Lucknow,109.7,null,33.2,9.9,2.1,36.9,29.3,1907,0
Ahmedabad,67.9,114.6,59.0,55.3,22.2,39.2,null,1381,407
Kolkata,64.4,115.6,40.4,8.4,0.8,30.7,18.4,759,759
Jorapokhar,64.2,149.7,9.3,33.6,0.7,32.4,7.0,379,894
Brajrajnagar,64.1,124.2,16.5,9.8,1.8,16.6,36.7,753,765
Guwahati,63.7,116.6,13.6,14.7,0.7,25.1,11.1,501,502
Talcher,61.4,165.8,13.8,28.5,1.8,17.2,11.5,741,746


In [0]:
gold_category_days = (
    silver
    .filter(F.col("AQI_Bucket").isNotNull())
    .groupBy("City", "Year", "AQI_Bucket")
    .agg(F.count("AQI_Bucket").alias("days"))
    .orderBy("City", "Year", "AQI_Bucket")
)

gold_category_days.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_aqi_category_days")
display(gold_category_days)

City,Year,AQI_Bucket,days
Ahmedabad,2015,Good,1
Ahmedabad,2015,Moderate,72
Ahmedabad,2015,Poor,74
Ahmedabad,2015,Satisfactory,8
Ahmedabad,2015,Severe,51
Ahmedabad,2015,Very Poor,57
Ahmedabad,2016,Moderate,12
Ahmedabad,2016,Poor,65
Ahmedabad,2016,Satisfactory,3
Ahmedabad,2016,Severe,20


In [0]:
display(spark.sql("SELECT SUM(days) AS total_days FROM workspace.default.gold_aqi_category_days"))

total_days
24850
